In [1]:
!rm -rf /kaggle/working/ML-for-Wireless-Security-Threats-Identification
!git clone https://github.com/duynhat326/ML-for-Wireless-Security-Threats-Identification.git /kaggle/working/ML-for-Wireless-Security-Threats-Identification

Cloning into '/kaggle/working/ML-for-Wireless-Security-Threats-Identification'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 128 (delta 57), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 582.48 KiB | 10.04 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [2]:
import os, sys, time
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT", "/kaggle/working/ML-for-Wireless-Security-Threats-Identification"))
sys.path.insert(0, str(ROOT))
RESULTS_DIRECTORY, SAVED_MODEL_DIRECTORY = Path("results"), Path("saved_model")
RESULTS_DIRECTORY.mkdir(exist_ok=True)
SAVED_MODEL_DIRECTORY.mkdir(exist_ok=True)
import numpy as np
from shared.config import LABELS, SEED
from shared.data_loader import load_splits
from shared.evaluator import measure_latency_milliseconds, save_json, save_results
from shared.tracer import build_alerts, save_bundle
import lightgbm as lgb

In [3]:
splits = load_splits()
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
start_time = time.perf_counter()
model.fit(
    splits["features_train"], splits["labels_train"],
    eval_set=[(splits["features_val"], splits["labels_val"])],
    callbacks=[lgb.early_stopping(30, verbose=True), lgb.log_evaluation(50)],
)
train_time = time.perf_counter() - start_time

Training until validation scores don't improve for 30 rounds
[50]	valid_0's multi_logloss: 0.15984
Early stopping, best iteration is:
[61]	valid_0's multi_logloss: 0.155372


In [4]:
predictions = model.predict(splits["features_test"])
latency = measure_latency_milliseconds(model.predict, splits["features_test"])
metrics = save_results(RESULTS_DIRECTORY, "lightgbm", splits["labels_test"], predictions, train_time, latency,
                       (splits["features_train"].columns, model.feature_importances_))
save_bundle(SAVED_MODEL_DIRECTORY / "model.pkl", model, splits["schema"])

import pandas as pd
print(pd.read_csv("results/per_class_report.csv").to_string(index=False))

           class  precision   recall  f1-score   support
          Normal   0.774241 0.998048  0.872013  136754.0
          Deauth   0.997098 1.000000  0.998547    5842.0
           Disas   1.000000 1.000000  1.000000   11270.0
       (Re)Assoc   1.000000 0.656174  0.792398     826.0
         RogueAP   1.000000 1.000000  1.000000     197.0
           Krack   1.000000 1.000000  1.000000    7499.0
           Kr00k   0.000000 0.000000  0.000000   28771.0
             SSH   1.000000 0.660684  0.795677    1783.0
          Botnet   1.000000 0.351652  0.520329    8534.0
         Malware   0.999592 0.248252  0.397728   19742.0
   SQL_Injection   0.044776 0.030380  0.036199     395.0
            SSDP   0.978683 1.000000  0.989227  824978.0
       Evil_Twin   1.000000 0.862003  0.925888   15725.0
Website_spoofing   0.999946 0.914364  0.955242   60769.0
       macro avg   0.842453 0.694397  0.734518 1123085.0
    weighted avg   0.930872 0.948305  0.932369 1123085.0


In [5]:
from shared.tracer import ScikitLearnPredictor
probabilities = ScikitLearnPredictor(model, splits["schema"]).predict_proba(splits["features_test"])
alerts = build_alerts(splits["metadata_test"], probabilities.argmax(axis=1), probabilities.max(axis=1))
save_json(alerts, RESULTS_DIRECTORY / "trace_results.json")
print(metrics)
print(len(alerts), "incidents")

{'model': 'lightgbm', 'accuracy': 0.9483048923278291, 'macro_f1': 0.7345176966714442, 'weighted_f1': 0.9323690334873015, 'train_time_seconds': 361.6752943040001, 'predict_latency_milliseconds': 0.013915360000055443}
98 incidents
